In [ ]:
# Imports for Cross-Encoder models
import os, gc, time, wandb, random, logging

from typing import Any, Sequence

from datetime import datetime
from zoneinfo import ZoneInfo

import numpy as np
import polars as pl

from dotenv import load_dotenv

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import GroupKFold

from umap import UMAP
from umap.utils import disconnected_vertices

from hdbscan import HDBSCAN

from sentence_transformers import SentenceTransformer
from transformers import (
    PreTrainedTokenizer, PreTrainedTokenizerFast,
    AutoTokenizer, AutoModelForSequenceClassification
)

from transformers import logging as hf_logging
from transformers.utils.logging import disable_progress_bar

2026-06-19 18:47:06.079530: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1781894826.246499      58 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1781894826.296792      58 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1781894826.693996      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781894826.694028      58 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1781894826.694030      58 computation_placer.cc:177] computation placer alr

In [ ]:
# Load datasets and encode label as `label_id`
TRAIN_PATH = "../../data/train.csv"
TEST_PATH = "../../data/test.csv"

LABEL_COL = "answer"
LABEL2ID = {"A": 0, "B": 1, "C": 2, "D": 3, "E": 4}

train_data = pl.read_csv(TRAIN_PATH).with_row_index("_idx")
test_data = pl.read_csv(TEST_PATH).with_row_index("_idx")
train_data = train_data.with_columns(
    pl.col(LABEL_COL).replace(LABEL2ID).cast(pl.Int8).alias("label_id")
)

ids = train_data["id"].to_list()
prompts = train_data["prompt"].to_list()

In [ ]:
# Configure training parameters, data, models, device, and seeds for reproducibility
EPOCHS = 7
N_SPLITS = 5

BATCH_SIZE = 8
ACCUMULATION_STEPS = 4

MAX_TOKEN_LENGTH = 384

ID_COL = "id"
QUESTION_COL = "prompt"
OPTION_COLS  = ["A", "B", "C", "D", "E"]

EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

RANK_MODEL = "cross-encoder/ms-marco-MiniLM-L12-v2"
# RANK_MODEL = "cross-encoder/ms-marco-MiniLM-L2-v2"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SEED = 42

def set_seed(seed: int):
    
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

set_seed(SEED)

# Create directory to store saved models
BEST_DIR = "/kaggle/working/best-cv-models"
os.makedirs(BEST_DIR, exist_ok=True)

# Disable Rust parallelism for tokenizers
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# Configure logging levels to hide model-loading report
hf_logging.set_verbosity_error()

logging.getLogger("transformers").setLevel(logging.ERROR)
logging.getLogger("huggingface_hub").setLevel(logging.ERROR)
logging.getLogger("sentence_transformers").setLevel(logging.ERROR)

disable_progress_bar()

# Configure WandB info
ist_now = datetime.now(ZoneInfo("Asia/Kolkata"))
WANDB_RUN_NAME = f"{RANK_MODEL.split('/')[-1]}_{ist_now:%Y-%m-%d_%H-%M-%S}_IST"

load_dotenv()

wandb.login(key=os.getenv("WANDB_API_KEY"))

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: spandanjit2005 to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [5]:
# Define DataLoader seeding function, and Generator for reproducibility
def seed_worker(worker_id: int = 42):
    
    worker_seed = SEED + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)

train_generator = torch.Generator()
train_generator.manual_seed(SEED)
val_generator = torch.Generator()
val_generator.manual_seed(SEED)

test_generator = torch.Generator()
test_generator.manual_seed(SEED)

In [6]:
# Procompute context for whole train and test data
def add_context(df: pl.DataFrame) -> pl.DataFrame:
    return df.with_columns(
        pl.struct([QUESTION_COL] + OPTION_COLS).map_elements(
            lambda row: "Question: " + str(row[QUESTION_COL]) + " Options: " + " | ".join(
                f"{c}) {str(row[c])}" for c in OPTION_COLS
            ),
            return_dtype=pl.Utf8
        ).alias("context")
    )

train_data = add_context(train_data)
test_data = add_context(test_data)

In [7]:
# Define PairMCQDDataset class
class PairMCQDDataset(Dataset):
    
    def __init__(self, df: pl.DataFrame, tokenizer: Any, max_length: int = 384) -> None:        
        texts1 = []
        texts2 = []
        labels = []

        for row in df.iter_rows(named=True):
            context = row["context"]
            
            for c in OPTION_COLS:
                texts1.append(context)
                texts2.append(f"{c}) {str(row[c])}")
                labels.append(1.0 if c == row[LABEL_COL] else 0.0)

        enc = tokenizer(
            texts1,
            texts2,
            truncation=True,
            padding=True,
            max_length=max_length,
            return_tensors="pt"
        )

        self.encodings = enc
        self.labels = torch.tensor(labels, dtype=torch.float32)

    def __len__(self) -> int:
        return len(self.labels)

    def __getitem__(self, idx: int) -> dict[str, torch.Tensor]:
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = self.labels[idx]
        
        return item

In [8]:
# Define BatchMCQDataset class
class BatchMCQDataset(Dataset):
    
    def __init__(self, df: pl.DataFrame, has_labels: bool = True) -> None:
        self.df = df
        self.has_labels = has_labels

    def __len__(self) -> int:
        return len(self.df)

    def __getitem__(self, idx: int) -> dict[str, Any]:
        row = self.df.row(idx, named=True)
        
        item: dict[str, Any] = {
            "prompt": str(row["prompt"]),
            "choices": [str(row[c]) for c in OPTION_COLS]
        }
        
        if self.has_labels:
            item["labels"] = row["label_id"] 
            
        return item

In [ ]:
# Define custom collate function for dynamic padding for BatchMCQDataset
class BatchMCQCollateFn:
    def __init__(self, tokenizer: PreTrainedTokenizer | PreTrainedTokenizerFast | Any) -> None:
        self.tokenizer = tokenizer

    def __call__(self, batch: list[dict[str, Any]]) -> dict[str, torch.Tensor]:
        prompts = []
        choices = []
        labels = []
        
        for item in batch:
            prompts.extend([item["prompt"]] * len(OPTION_COLS))
            choices.extend(item["choices"])
            
            if "labels" in item: 
                labels.append(item["labels"])
                
        # Tokenize the whole batch at once with dynamic padding
        enc = self.tokenizer(
            prompts,
            choices,
            truncation=True,
            padding=True,
            max_length=MAX_TOKEN_LENGTH,
            return_tensors="pt"
        )
        
        B = len(batch)
        C = len(OPTION_COLS)
        
        output = {
            "input_ids": enc["input_ids"].view(B, C, -1),
            "attention_mask": enc["attention_mask"].view(B, C, -1)
        }
        
        if "token_type_ids" in enc:
            output["token_type_ids"] = enc["token_type_ids"].view(B, C, -1)
            
        if labels:
            output["labels"] = torch.tensor(labels, dtype=torch.long)
            
        return output

In [11]:
# Define function to compute Mean Average Precision@3 (MAP@3)
def compute_map3(scores: np.ndarray | Sequence, df: pl.DataFrame) -> float:
    scores = np.asarray(scores).reshape(len(df), 5)
    true = df["label_id"].to_numpy()

    top3 = np.argsort(-scores, axis=1)[:, :3]
    hit = top3 == true[:, None]
    ranks = np.where(hit.any(axis=1), hit.argmax(axis=1) + 1, 0)

    out = np.zeros_like(ranks, dtype=float)
    np.divide(1.0, ranks, out=out, where=ranks > 0)
    
    return float(out.mean())

In [13]:
# Define function to compute out-of-fold probabilities for PairMCQDataset
def compute_pair_oof_probs(model: nn.Module, loader: DataLoader, DEVICE: torch.device | str) -> np.ndarray:
    model.eval()

    start_idx = 0
    num_samples = len(loader.dataset)
    all_probs = torch.zeros((num_samples, 5), device=DEVICE)

    with torch.no_grad():
        for batch in loader:
            inputs = {
                k: v.to(DEVICE, non_blocking=True)
                for k, v in batch.items()
                if k != "labels"
            }
            
            logits = model(**inputs).logits.squeeze(-1)

            batch_probs = F.softmax(logits, dim=1)
            
            batch_size = batch_probs.size(0)
            all_probs[start_idx : start_idx + batch_size] = batch_probs
            start_idx += batch_size

    return all_probs.cpu().numpy()

In [14]:
# Define function to compute out-of-fold probabilities for BatchMCQDataset
def compute_batch_oof_probs(model: nn.Module, loader: DataLoader, DEVICE: torch.device | str) -> np.ndarray:
    model.eval()
    
    start_idx = 0
    num_samples = len(loader.dataset)
    all_probs = torch.zeros((num_samples, 5), device=DEVICE)

    with torch.no_grad():
        for batch in loader:
            inputs = {
                k: v.to(DEVICE, non_blocking=True)
                for k, v in batch.items()
                if k != "labels"
            }

            B, C, L = inputs["input_ids"].shape
            flat_inputs = {k: v.view(B * C, L) for k, v in inputs.items()}
            
            logits = model(**flat_inputs).logits.squeeze(-1).view(B, C)

            batch_probs = F.softmax(logits, dim=1)
            
            batch_size = batch_probs.size(0)
            all_probs[start_idx : start_idx + batch_size] = batch_probs
            start_idx += batch_size

    return all_probs.cpu().numpy()

In [15]:
# Get embeddings for dimensionality reduction and clustering
embed_model = SentenceTransformer(EMBED_MODEL).to(DEVICE)
embeddings = embed_model.encode(prompts, batch_size=32)

del embed_model
gc.collect()

2178

In [16]:
# Reduce high-dimensional embeddings to 10 dimensions for clustering
umap_model = UMAP(
    n_components=10,
    n_neighbors=30,
    min_dist=0.0,
    metric="cosine",
    n_jobs=1,
    random_state=SEED
)

umap_data = umap_model.fit_transform(embeddings)

disconnected = disconnected_vertices(umap_model)
valid_indices = np.where(~disconnected)[0]

umap_data = umap_data[valid_indices]

valid_ids = np.array(ids)[valid_indices]
valid_prompts = np.array(prompts)[valid_indices]

# Find clusters from 10 dimensional embeddings for better cross-validation
clusterer_umap = HDBSCAN(
    min_cluster_size=7,
    min_samples=5,
    metric='euclidean',
    cluster_selection_method='eom',
    prediction_data=True
)

cluster_labels_umap = clusterer_umap.fit_predict(umap_data)

valid_clusters = np.array(cluster_labels_umap)

In [17]:
# Add cluster information to train datafrane
cluster_data = pl.DataFrame({
    "id": valid_ids,
    "prompt": valid_prompts,
    "cluster": valid_clusters,
})

train_data = train_data.join(
    cluster_data.select("id", "cluster"),
    on="id",
    how="left"
)

In [ ]:
# # Configure CV constructor, dummy features, and Tokenizer
# gkf = GroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

# X = np.zeros(len(train_data))
# y = np.zeros(len(train_data))
# groups = train_data["cluster"].fill_null(-1).to_numpy()

# tokenizer = AutoTokenizer.from_pretrained(RANK_MODEL)

# oof_probs = np.zeros((len(train_data), 5), dtype=np.float32)
# best_fold_paths = []

# # Configure WandB logging
# run = wandb.init(
#     entity="24f2005537-dl-genai-project",
#     project="dl-genai-project",
#     name=WANDB_RUN_NAME,
#     config={
#         "model": {
#             "name": RANK_MODEL,
#             "architecture": "BERT",
#             "total_parameters": "~33.4M",
#             "transformer_layers": 12,
#             "hidden_size": 384,
#             "max_sequence_length": 512,
#         },
#         "training": {
#             "splits": N_SPLITS,
#             "epochs": EPOCHS,
#             "optimizer": "AdamW",
#             "lr": 2e-5,
#             "loss": "BCEWithLogitsLoss",
#             "scheduler": None
#         },
#         "data": {
#             "batch_size": 32,
#             "max_length": 384
#         }
#     }
# )

# # Start iterating through {N_SPLITS}
# for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y, groups)):

#     fold_start = time.perf_counter()
#     print(f"FOLD {fold+1}/{N_SPLITS} | "
#           f"Train Set Size: {len(train_idx)} | Val Set Size: {len(val_idx)}\n")

#     train_fold = train_data[train_idx]
#     val_fold = train_data[val_idx]

#     # Configure DataLoaders for current fold
#     train_loader = DataLoader(
#         PairMCQDDataset(
#             train_fold, 
#             tokenizer=tokenizer
#         ),
#         batch_size=32,
#         shuffle=True,
#         num_workers=4,
#         pin_memory=True,
#         worker_init_fn=seed_worker,
#         generator=train_generator
#     )

#     val_loader = DataLoader(
#         PairMCQDDataset(
#             val_fold, 
#             tokenizer=tokenizer
#         ),
#         batch_size=32,
#         shuffle=False,
#         num_workers=4,
#         pin_memory=True,
#         worker_init_fn=seed_worker,
#         generator=val_generator
#     )

#     model = AutoModelForSequenceClassification.from_pretrained(RANK_MODEL, num_labels=1)
    
#     if torch.cuda.device_count() > 1:
#         model = nn.DataParallel(model, device_ids=[0, 1])
#     model = model.to(DEVICE)

#     optimizer = optim.AdamW(
#         model.module.parameters() if isinstance(model, nn.DataParallel) else model.parameters(),
#         lr=2e-5,
#         weight_decay=0.025
#     )
#     criterion = nn.BCEWithLogitsLoss()
    
#     best_val_map3 = -1.0
#     best_path = os.path.join(BEST_DIR, f"f_{fold+1}_best.pt")
#     best_fold_paths.append(best_path)

#     # Srart training for {EPOCHS} in current fold
#     for epoch in range(EPOCHS):
#         epoch_start = time.perf_counter()

#         model.train()
#         train_loss_sum = 0.0

#         for batch in train_loader:
#             labels = batch["labels"].to(DEVICE, non_blocking=True)
#             inputs = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items() if k != "labels"}

#             optimizer.zero_grad()
#             logits = model(**inputs).logits.squeeze(-1)
#             loss = criterion(logits, labels)

#             loss.backward()
#             optimizer.step()

#             train_loss_sum += loss.item() * labels.size(0)

#         train_loss = train_loss_sum / len(train_fold)

#         model.eval()
#         val_loss_sum = 0.0
#         val_probs_all = []

#         # Compute validation scores for current epoch
#         with torch.no_grad():
#             for batch in val_loader:
#                 labels = batch["labels"].to(DEVICE, non_blocking=True)
#                 inputs = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items() if k != "labels"}

#                 logits = model(**inputs).logits.squeeze(-1)
#                 loss = criterion(logits, labels)

#                 val_loss_sum += loss.item() * labels.size(0)
#                 val_probs_all.extend(logits.detach().cpu().numpy().tolist())

#         val_loss = val_loss_sum / len(val_fold)
#         val_map3 = compute_map3(val_probs_all, val_fold)

#         epoch_time = time.perf_counter() - epoch_start

#         run.log({
#             "epoch_secs": round(epoch_time, 4),
#             "train_loss": train_loss,
#             "val_loss": val_loss, "val_map3": val_map3
#         })
#         print(
#             f"Fold: {fold+1}/{N_SPLITS} Epoch: {epoch+1}/{EPOCHS} Time: {epoch_time:.2f}s\n"
#             f"\tTrain Loss: {train_loss:.6f}\n"
#             f"\tVal Loss: {val_loss:.6f} | Val MAP@3: {val_map3:.6f}"
#         )

#         # Save best model per fold
#         if val_map3 > best_val_map3:
#             best_val_map3 = val_map3    
#             state = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
#             torch.save(state, best_path)

#     best_ce = AutoModelForSequenceClassification.from_pretrained(RANK_MODEL, num_labels=1)
#     state = torch.load(best_path, map_location="cpu")
#     best_ce.load_state_dict(state)
    
#     if torch.cuda.device_count() > 1:
#         best_ce = nn.DataParallel(best_ce, device_ids=[0, 1])
    
#     best_ce = best_ce.to(DEVICE)

#     best_val_probs = compute_pair_oof_probs(best_ce, val_loader, DEVICE)
#     oof_probs[val_idx] = best_val_probs.reshape(len(val_fold), 5)
    
#     fold_time = time.perf_counter() - fold_start

#     run.log({
#         "fold_secs": round(fold_time, 4),
#         "best_val_map3": best_val_map3
#     })
#     print(
#         f"\nFold {fold+1} completed in {int(fold_time // 60)} min {fold_time % 60:.2f}s"
#         f" | Best Val MAP@3: {best_val_map3:.6f}\n"
#     )

#     del best_ce, optimizer
#     torch.cuda.empty_cache()
#     gc.collect()

# run.finish()
# oof_map3 = compute_map3(oof_probs, train_data)
# print(f"\nOut-of-fold MAP@3 score: {oof_map3:.8f}")

In [ ]:
# Configure CV constructor, dummy features, tokenizer, base model, and collate function
gkf = GroupKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

X = np.zeros(len(train_data))
y = np.zeros(len(train_data))
groups = train_data["cluster"].fill_null(-1).to_numpy()

tokenizer = AutoTokenizer.from_pretrained(RANK_MODEL)
base_model = AutoModelForSequenceClassification.from_pretrained(RANK_MODEL, num_labels=1)

base_model.save_pretrained(BEST_DIR)
tokenizer.save_pretrained(BEST_DIR)
del base_model

batch_mcq_collate_fn = BatchMCQCollateFn(tokenizer)

oof_probs = np.zeros((len(train_data), 5), dtype=np.float32)
best_fold_paths = []

# Configure WandB logging
run = wandb.init(
    entity="24f2005537-dl-genai-project",
    project="dl-genai-project",
    name=WANDB_RUN_NAME,
    config={
        "model": {
            "name": RANK_MODEL,
            "architecture": "BERT",
            "total_parameters": "~33.4M",
            "transformer_layers": 12,
            "max_sequence_length": 512,
        },
        "training": {
            "splits": N_SPLITS,
            "epochs": EPOCHS,
            "optimizer": "AdamW",
            "lr": 2e-5,
            "loss": "CrossEntropyLoss",
            "scheduler": None
        },
        "data": {
            "batch_size": BATCH_SIZE,
            "max_length": MAX_TOKEN_LENGTH
        }
    }
)

# Start iterating through {N_SPLITS}
for fold, (train_idx, val_idx) in enumerate(gkf.split(X, y, groups)):

    fold_start = time.perf_counter()
    print(f"FOLD {fold+1}/{N_SPLITS} | "
          f"Train Set Size: {len(train_idx)} | Val Set Size: {len(val_idx)}\n")

    train_fold = train_data[train_idx]
    val_fold = train_data[val_idx]

    # Configure DataLoaders for current fold
    train_loader = DataLoader(
        BatchMCQDataset(
            train_fold,
            has_labels=True
        ),
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=4,
        pin_memory=True,
        worker_init_fn=seed_worker,
        generator=train_generator,
        collate_fn=batch_mcq_collate_fn
    )

    val_loader = DataLoader(
        BatchMCQDataset(
            val_fold,
            has_labels=True
        ),
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=4,
        pin_memory=True,
        worker_init_fn=seed_worker,
        generator=val_generator,
        collate_fn=batch_mcq_collate_fn
    )

    # Configure model for current fold
    model = AutoModelForSequenceClassification.from_pretrained(BEST_DIR, num_labels=1)
    
    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model, device_ids=[0, 1])
    model = model.to(DEVICE)

    # Configure active parameters, optimizer, and loss function for current fold
    active_params = [
        p for p in (model.module.parameters() if isinstance(model, nn.DataParallel) else model.parameters()) 
        if p.requires_grad
    ]
    optimizer = optim.AdamW(
        active_params,
        lr=2e-5,
        weight_decay=0.01
    )
    criterion = nn.CrossEntropyLoss()

    # Configure best validation score and best model path for current fold
    best_val_map3 = -1.0
    best_path = os.path.join(BEST_DIR, f"f_{fold+1}_best.pt")
    best_fold_paths.append(best_path)

    # Start training loop for current fold
    for epoch in range(EPOCHS):
        epoch_start = time.perf_counter()

        model.train()
        train_loss_sum = 0.0

        optimizer.zero_grad()
        
        for step, batch in enumerate(train_loader):
            labels = batch["labels"].to(DEVICE, non_blocking=True)
            inputs = {
                k: v.to(DEVICE, non_blocking=True)
                for k, v in batch.items()
                if k != "labels"
            }
        
            B, C, L = inputs["input_ids"].shape
            flat_inputs = {k: v.view(B * C, L) for k, v in inputs.items()}
        
            # Forward pass
            train_logits = model(**flat_inputs).logits.squeeze(-1).view(B, C)

            # Calculate standard loss
            loss = criterion(train_logits, labels)

            # Scale the loss
            steps_to_accumulate = min(ACCUMULATION_STEPS, len(train_loader) - (step // ACCUMULATION_STEPS) * ACCUMULATION_STEPS)
            scaled_loss = loss / steps_to_accumulate
            scaled_loss.backward()

            # Optimizer step and zero gradients when `ACCUMULATION_STEPS` is reached, or at end of epoch
            if (step + 1) % ACCUMULATION_STEPS == 0 or (step + 1) == len(train_loader):
                optimizer.step()
                optimizer.zero_grad()
        
            train_loss_sum += loss.item() * B

        train_loss = train_loss_sum / len(train_fold)

        model.eval()
        val_loss_sum = 0.0
        
        val_start_idx = 0
        num_val_samples = len(val_loader.dataset)
        all_val_probs = torch.zeros((num_val_samples, 5), device=DEVICE)

        # Compute validation scores for current epoch
        with torch.no_grad():
            for batch in val_loader:
                labels = batch["labels"].to(DEVICE, non_blocking=True)
                inputs = {
                    k: v.to(DEVICE, non_blocking=True)
                    for k, v in batch.items()
                    if k != "labels"
                }

                B, C, L = inputs["input_ids"].shape
                flat_inputs = {k: v.view(B * C, L) for k, v in inputs.items()}

                val_logits = model(**flat_inputs).logits.squeeze(-1).view(B, C)

                val_probs = F.softmax(val_logits, dim=1)
                
                loss = criterion(val_logits, labels)
                val_loss_sum += loss.item() * B
                
                batch_size = val_probs.size(0)
                all_val_probs[val_start_idx : val_start_idx + batch_size] = val_probs
                val_start_idx += batch_size

        all_val_probs = all_val_probs.cpu().numpy()
        
        val_loss = val_loss_sum / num_val_samples
        val_map3 = compute_map3(all_val_probs, val_fold)

        epoch_time = time.perf_counter() - epoch_start

        run.log({
            "epoch_secs": round(epoch_time, 4),
            "train_loss": train_loss,
            "val_loss": val_loss, "val_map3": val_map3
        })
        print(
            f"Fold: {fold+1}/{N_SPLITS} Epoch: {epoch+1}/{EPOCHS} Time: {epoch_time:.2f}s\n"
            f"\tTrain Loss: {train_loss:.6f}\n"
            f"\tVal Loss: {val_loss:.6f} | Val MAP@3: {val_map3:.6f}"
        )

        # Save best model per fold
        if val_map3 > best_val_map3:
            best_val_map3 = val_map3    
            state = model.module.state_dict() if isinstance(model, nn.DataParallel) else model.state_dict()
            torch.save(state, best_path)

    del model
    gc.collect()

    # Load saved best model for current fold to compute validation score
    best_ce = AutoModelForSequenceClassification.from_pretrained(BEST_DIR, num_labels=1)
    state = torch.load(best_path, map_location="cpu")
    best_ce.load_state_dict(state)

    if torch.cuda.device_count() > 1:
        best_ce = nn.DataParallel(best_ce, device_ids=[0, 1])

    best_ce = best_ce.to(DEVICE)

    best_val_probs = compute_batch_oof_probs(best_ce, val_loader, DEVICE)
    oof_probs[val_idx] = best_val_probs

    fold_time = time.perf_counter() - fold_start

    run.log({
        "fold_secs": round(fold_time, 4),
        "best_val_map3": best_val_map3
    })
    print(
        f"\nFold {fold+1} completed in {int(fold_time // 60)} min {fold_time % 60:.2f}s"
        f" | Best Val MAP@3: {best_val_map3:.6f}\n"
    )

    del best_ce, optimizer
    torch.cuda.empty_cache()
    gc.collect()

run.finish()
oof_map3 = compute_map3(oof_probs, train_data)
print(f"\nOut-of-fold MAP@3 score: {oof_map3:.8f}")

OOF to beat: 
- 0.56400000 [0.73399] (dynamic init + grad accumulation)

In [ ]:
# # Inference code for a given cross-encoder model trained with BCEWithLogitsLoss
# test_scores = np.zeros((len(test_data), 5))

# tmp = test_data.with_columns(pl.lit("A").alias("answer"))

# test_loader = DataLoader(
#     PairMCQDDataset(
#         tmp, 
#         tokenizer=tokenizer
#     ),
#     batch_size=32,
#     shuffle=False,
#     num_workers=4,
#     pin_memory=True,
#     worker_init_fn=seed_worker,
#     generator=test_generator
# )

# for fold, best_path in enumerate(best_fold_paths):
#     model = AutoModelForSequenceClassification.from_pretrained(BEST_DIR, num_labels=1)
#     state = torch.load(best_path, map_location="cpu")
#     model.load_state_dict(state)

#     if torch.cuda.device_count() > 1:
#         model = nn.DataParallel(model, device_ids=[0, 1])
#     model = model.to(DEVICE)

#     model.eval()

#     test_start_idx = 0
#     num_test_samples = len(test_loader.dataset)
#     all_test_probs = torch.zeros(num_test_samples, device=DEVICE)

#     with torch.no_grad():
#         for batch in test_loader:
#             inputs = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items() if k != "labels"}
            
#             test_logits = model(**inputs).logits.squeeze(-1) 
            
#             test_probs = torch.sigmoid(test_logits)

#             batch_size = test_probs.size(0)
#             all_test_probs[test_start_idx : test_start_idx + batch_size] = test_probs
#             test_start_idx += batch_size

#     fold_probs = all_test_probs.view(-1, 5).cpu().numpy()
    
#     test_scores += fold_probs / len(best_fold_paths)
    
#     del model
#     torch.cuda.empty_cache()
#     gc.collect()

# top3_idx = np.argsort(-test_scores, axis=1)[:, :3]
# pred_strings = [" ".join(OPTION_COLS[i] for i in row) for row in top3_idx]

# submission = pl.DataFrame({"ID": test_data[ID_COL], "Prediction": pred_strings})
# submission.write_csv("submission.csv")
# print(submission.sample(5))

In [ ]:
# Inference code for a given cross-encoder model trained with CrossEntropyLoss
test_scores = np.zeros((len(test_data), 5))

test_loader = DataLoader(
    BatchMCQDataset(
        test_data,
        has_labels=False
    ),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    pin_memory=True,
    worker_init_fn=seed_worker,
    generator=test_generator,
    collate_fn=batch_mcq_collate_fn
)

for fold, best_path in enumerate(best_fold_paths):
    model = AutoModelForSequenceClassification.from_pretrained(BEST_DIR, num_labels=1)
    state = torch.load(best_path, map_location="cpu")
    model.load_state_dict(state)

    if torch.cuda.device_count() > 1:
        model = nn.DataParallel(model, device_ids=[0, 1])
    model = model.to(DEVICE)

    model.eval()

    test_start_idx = 0
    num_test_samples = len(test_loader.dataset)
    all_test_probs = torch.zeros((num_test_samples, 5), device=DEVICE)

    with torch.no_grad():
        for batch in test_loader:
            inputs = {k: v.to(DEVICE, non_blocking=True) for k, v in batch.items()}
    
            B, C, L = inputs["input_ids"].shape
    
            flat_inputs = {k: v.view(B * C, L) for k, v in inputs.items()}
            test_logits = model(**flat_inputs).logits.squeeze(-1).view(B, C)
    
            test_probs = F.softmax(test_logits, dim=1)
    
            batch_size = test_probs.size(0)
            all_test_probs[test_start_idx : test_start_idx + batch_size] = test_probs
            test_start_idx += batch_size

    all_test_probs = all_test_probs.cpu().numpy()

    test_scores += all_test_probs / len(best_fold_paths)

    del model
    torch.cuda.empty_cache()
    gc.collect()

top3_idx = np.argsort(-test_scores, axis=1)[:, :3]
pred_strings = [" ".join(OPTION_COLS[i] for i in row) for row in top3_idx]

submission = pl.DataFrame({"ID": test_data[ID_COL], "Prediction": pred_strings})
submission.write_csv("submission.csv")
print(submission.sample(5))

shape: (5, 2)
┌─────┬────────────┐
│ ID  ┆ Prediction │
│ --- ┆ ---        │
│ i64 ┆ str        │
╞═════╪════════════╡
│ 83  ┆ D B E      │
│ 152 ┆ B D E      │
│ 200 ┆ C A D      │
│ 61  ┆ C A D      │
│ 354 ┆ E B C      │
└─────┴────────────┘
